# 3 비교모델 60회 — EfficientAT · PANNs · CED-Tiny × 정답 라벨만 × 정답 200·500·1,000·전량 × seed 1~5

**L4 로 실행하십시오.** [셀 1] 이 Blackwell(sm_100·sm_120)이면 멈춥니다. GPU 가 없으면 CPU 로 돕니다(시간 제한 없음).

**'모두 실행' 한 번으로 끝까지 돕니다.** 세션이 끊기면 다시 '모두 실행' — 끝난 회차는 건너뜁니다(seed 1 의 12회는 10-26 새벽에 끝남).
`2 Mark5.3 본실험 40회` 노트북과 동시에 돌려도 됩니다. 결과물 4개는 그 노트북의 마지막 셀이 이 60회가 끝나길 기다렸다가 만듭니다.

- 비교모델은 지식 융합 구조가 아니므로 우리 가짜 정답(plc)을 쓰지 않습니다 — 정답 라벨만 학습합니다(2026-10-26 확정).
- 반복은 제안모델처럼 seed 1~5 입니다(2026-10-26). 옵티마이저는 Adam, 학습률은 1e-4 고정입니다(run_tag `_adam`).
- PANNs 는 코랩 기본 환경에 없는 `torchlibrosa` 가 필요해 [셀 3b] 에서 설치합니다.
- 오디오는 `Mark5.2` 폴더의 것을 씁니다(`--data_root`).

In [ ]:
#========================================================================
# [셀 1] 런타임 확인 — L4 권장 · Blackwell(sm_100·120)이면 멈춤 · CPU 는 허용(멈추지 않음)
#========================================================================
import subprocess, sys
_probe = subprocess.run([sys.executable, "-c",
    "import os, torch\n"
    "ok = torch.cuda.is_available()\n"
    "cap = torch.cuda.get_device_capability(0) if ok else None\n"
    "print('GPU', torch.cuda.get_device_name(0) if ok else '없음 — CPU 로 돈다')\n"
    "print('SM', cap[0] * 10 + cap[1] if cap else 'cpu')\n"
    "print('torch', torch.__version__, '| CUDA', torch.version.cuda, '| CPU 코어', os.cpu_count())"],
    capture_output=True, text=True)
print(_probe.stdout, _probe.stderr[-2000:])
_sm = next((l.split()[1] for l in _probe.stdout.splitlines() if l.startswith("SM ")), "cpu")
if _sm in ("100", "120"):
    raise SystemExit("[STOP] Blackwell(sm_100/sm_120) 런타임입니다. L4 로 바꿔 다시 '모두 실행' 하십시오(2026-10-17 G4 계산 오류).")
!nvidia-smi --query-gpu=name,driver_version,compute_cap,memory.total --format=csv 2>/dev/null || echo "nvidia-smi 없음(CPU 런타임)"
!free -g | head -2

In [ ]:
#========================================================================
# [셀 2] 구글 드라이브 마운트
#========================================================================
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
#========================================================================
# [셀 3] 코드 판 확인 + 데이터 목록 — Mark5.2 의 목록과 같게 맞추고 행 수 확인
#========================================================================
import csv, json, os, re, time
EXP = "/content/drive/MyDrive/NewProject/codes/experiments"
DATA_ROOT = f"{EXP}/Mark5.2"
REPOS = {"Baseline_EfficientAT_mn10": "efficientat_mn10", "Baseline_PANNs_CNN6": "panns_cnn6", "Baseline_CED_Tiny": "ced_tiny"}
NEED = {"common/naming.py": ['OPTIMIZER_NAME = "adam"', "def adam_lr"],   # [2026-10-25 / A안] Adam · 선정값 고정 · run_tag _adam
        "common/train_loop.py": ["optimizer = Adam("]}
for repo in REPOS:
    root = f"{EXP}/{repo}"
    miss = [(f, s) for f, ss in NEED.items() for s in ss
            if not os.path.isfile(f"{root}/{f}") or s not in open(f"{root}/{f}", encoding="utf-8").read()]
    if miss:
        raise SystemExit(f"[STOP] {repo} 코드가 10-25(A안, Adam) 판이 아닙니다: {miss}")
    for name in ("dataset_labeled.csv", "dataset_unlabeled.csv", "dataset_val.csv", "dataset_test.csv"):
        src, dst = f"{DATA_ROOT}/{name}", f"{root}/{name}"
        if open(src, "rb").read() != (open(dst, "rb").read() if os.path.isfile(dst) else b""):
            open(dst, "wb").write(open(src, "rb").read())
            print(f"[복사] {repo}/{name} <- Mark5.2")
    rows = {n: sum(1 for _ in csv.DictReader(open(f"{root}/{n}", encoding="utf-8-sig")))
            for n in ("dataset_labeled.csv", "dataset_unlabeled.csv", "dataset_val.csv", "dataset_test.csv")}
    if rows != {"dataset_labeled.csv": 2000, "dataset_unlabeled.csv": 2000, "dataset_val.csv": 430, "dataset_test.csv": 430}:
        raise SystemExit(f"[STOP] {repo} 데이터 목록 행 수 {rows}")
    print(f"{repo}: 코드 판 확인 · 데이터 목록 {rows}")

In [ ]:
#========================================================================
# [셀 3b] 패키지 — PANNs 전용 torchlibrosa(코랩 기본 환경에 없음). 2026-10-25 에는 이 셀이 없어 PANNs 회차가 시작 전에 실패했다
#========================================================================
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "torchlibrosa==0.1.0"], check=True)
_chk = subprocess.run([sys.executable, "-c", "import model_def, importlib.metadata as m; print('torchlibrosa', m.version('torchlibrosa'))"],
                      capture_output=True, text=True, cwd=f"{EXP}/Baseline_PANNs_CNN6")
print(_chk.stdout, _chk.stderr[-2000:])
if _chk.returncode != 0 or "torchlibrosa 0.1.0" not in _chk.stdout:
    raise SystemExit("[STOP] PANNs 모델 코드를 불러오지 못했습니다(torchlibrosa 0.1.0) — 위 오류 확인 뒤 다시 '모두 실행'")

In [ ]:
#========================================================================
# [셀 4] 계획 확인 — 저장소마다 20줄(정답 라벨만 × 정답 4수준 × seed 1~5), 태그는 Adam 선정값 고정(_adam_)
#========================================================================
import subprocess
B_RUN = "--phase main --conds sup --labeled_ns 0 1000 500 200 --seeds 1 2 3 4 5"


def plan(repo):
    p = subprocess.run(f"{sys.executable} run_experiments.py {B_RUN} --data_root {DATA_ROOT} --dry_run",
                       shell=True, capture_output=True, text=True, cwd=f"{EXP}/{repo}")
    rows = [l.strip() for l in p.stdout.splitlines() if "[실행]" in l or "[건너뜀]" in l]
    if p.returncode != 0 or len(rows) != 20 or not all("_adam_" in r for r in rows):
        print(p.stdout[-3000:]); print(p.stderr[-3000:])
        raise SystemExit(f"[STOP] {repo} 계획을 읽지 못했습니다(종료 코드 {p.returncode} · {len(rows)}줄, 기대 20줄)")
    return rows


for repo in REPOS:
    rows = plan(repo)
    print(f"{repo}: 남은 {sum('[실행]' in r for r in rows)} / 20")

In [ ]:
#========================================================================
# [셀 5] 비교모델 ★ 3 × 20회 — 정답 라벨만 × 정답 전량·1,000·500·200 × seed 1~5. 끝난 회차는 건너뛴다. 시간 제한 없음
#========================================================================
for repo in REPOS:
    print(f"\n######## {repo}  정답 라벨만 × 정답 전량·1,000·500·200 × seed 1~5", flush=True)
    !cd "{EXP}/{repo}" && python run_experiments.py --phase main --conds sup --labeled_ns 0 1000 500 200 --seeds 1 2 3 4 5 --data_root "{DATA_ROOT}"

In [ ]:
#========================================================================
# [셀 6] 확인 — 60회가 전부 끝났는가 (남았으면 멈춘다) · 모델·정답 수준별 test 4지표 5회 평균
#========================================================================
import statistics
left = {repo: [r for r in plan(repo) if "[실행]" in r] for repo in REPOS}
if any(left.values()):
    for repo, rows in left.items():
        for r in rows:
            print(repo, r)
    raise SystemExit(f"[STOP] 남은 회차 {sum(map(len, left.values()))}개 — 셀 5 출력의 [FAIL] 을 확인하고 다시 '모두 실행'")
METRICS = ("Accuracy", "Macro Precision", "Macro Recall", "Macro F1")


def read_summary(path):
    if not os.path.isfile(path):
        return None
    out = {}
    for line in open(path, encoding="utf-8"):
        for m in METRICS:
            if line.startswith(m + ","):
                out[m] = float(line.split(",", 1)[1])
    return out if len(out) == len(METRICS) else None


for repo, mtag in REPOS.items():
    print(f"\n{repo}")
    for l in (200, 500, 1000, None):
        vals = [r for s in range(1, 6)
                if (r := read_summary(f"{EXP}/{repo}/plots/performance_summary_{mtag}_sup_adam" + ("" if l is None else f"_l{l}") + f"_s{s}.csv"))]
        txt = " · ".join(f"{m} {statistics.mean(v[m] for v in vals):.4f}" for m in METRICS) if vals else "없음"
        print(f"  정답 {'전량' if l is None else l} (n={len(vals)}): {txt}")
print("\n60회 완료 — 결과물 4개는 '2 Mark5.3 본실험 40회' 노트북의 마지막 셀이 만듭니다.")